# 01. Data Quality

This notebook inspects audit anomalies, missing metadata, and cleaning-sensitive fields.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from notebooks.notebook_utils import *

set_plot_style()
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

corpus = load_corpus_stats()
meta = load_verdict_metadata()
mismatches = load_amar_mismatches()
print(corpus.shape, meta.shape, mismatches.shape)

## Missingness profile

In [ ]:
quality = pd.DataFrame({
    'corpus_stats_missing_pct': corpus.isna().mean().mul(100),
    'verdict_metadata_missing_pct': meta.isna().mean().mul(100),
}).fillna(0).sort_values('verdict_metadata_missing_pct', ascending=False)
quality.head(20).round(2)

In [ ]:
heatmap_data = quality.head(12)
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd')
plt.title('Top missingness rates (%)')
plt.tight_layout()

## Amar mismatch examples

In [ ]:
mismatches[['file_id', 'nomor_putusan', 'amar_json', 'amar_text']].head(15)

## Potential cleaning artifacts

In [ ]:
artifact_flags = corpus.assign(
    no_amar_section=~corpus['has_amar_section'].astype(bool),
    missing_panel=~corpus['has_panel_ketua'].astype(bool),
    missing_date=~corpus['has_tanggal_putusan'].astype(bool),
)
artifact_flags[['no_amar_section', 'missing_panel', 'missing_date']].mean().mul(100).round(2).to_frame('percent_of_corpus')